In [1]:
import pandas as pd
import torch
import os
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

In [2]:
model_name = "mistralai/Mistral-7B-v0.3"
input_file = "final_combined_dataset.csv"
output_file = "abstention_mistral_final.csv"
save_every = 50
batch_size = 10

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
     

In [4]:
df = pd.read_csv(input_file)

if os.path.exists(output_file):
    df_existing = pd.read_csv(output_file)
    start_idx = len(df_existing)
    print(f" Resuming from row {start_idx}")
else:
    start_idx = 0

In [5]:
def build_prompt(row):
    return f"""
You are an expert MCQ solver.

Question: {row['question']}

Options:
a) {row['option_a']}
b) {row['option_b']}
c) {row['option_c']}
d) {row['option_d']}

INSTRUCTIONS:
- Choose exactly one: a, b, c, d
- If uncertain → idk
- Always provide reasoning

OUTPUT FORMAT (STRICT, DO NOT DEVIATE):
<answer>a/b/c/d/idk</answer>
<reasoning>Give 2-3 line explanation</reasoning>

Only output these two tags. No extra text.
"""

In [6]:
def parse_output(output):
    final_answer = "idk"
    reasoning = ""

    try:
        ans_match = re.search(r"<answer>\s*(.*?)\s*</answer>", output, re.IGNORECASE)
        if ans_match:
            final_answer = ans_match.group(1).strip().lower()

        reason_match = re.search(r"<reasoning>\s*(.*?)\s*</reasoning>", output, re.IGNORECASE | re.DOTALL)
        if reason_match:
            reasoning = reason_match.group(1).strip()

    except:
        pass

    if final_answer not in ["a", "b", "c", "d", "idk"]:
        final_answer = "idk"

    if reasoning == "":
        final_answer = "idk"

    return final_answer, reasoning

In [7]:
for i in tqdm(range(start_idx, len(df), batch_size), desc="Running Inference"):

    batch = df.iloc[i:i+batch_size].copy()
    prompts = [build_prompt(row) for _, row in batch.iterrows()]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None
        )

    input_len = inputs.input_ids.shape[1]
    generated_tokens = outputs[:, input_len:]

    decoded = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    pred_answers = []
    pred_reasoning = []

    for output in decoded:
        ans, reason = parse_output(output)
        pred_answers.append(ans)
        pred_reasoning.append(reason)

    batch["pred_final_answer"] = pred_answers
    batch["pred_reasoning"] = pred_reasoning

    if os.path.exists(output_file):
        batch.to_csv(output_file, mode="a", index=False, header=False)
    else:
        batch.to_csv(output_file, index=False)

    print(f"Saved batch {i} to {i + len(batch)}")

print("Done! Fully robust pipeline completed.")

Running Inference:   0%|          | 1/353 [00:03<22:03,  3.76s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Saved batch 0 to 10


Running Inference:   1%|          | 2/353 [00:06<19:42,  3.37s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Saved batch 10 to 20


Running Inference:   1%|          | 3/353 [00:09<18:59,  3.25s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


...

Saved batch 3500 to 3510


Running Inference: 100%|█████████▉| 352/353 [19:48<00:06,  6.45s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Saved batch 3510 to 3520


Running Inference: 100%|██████████| 353/353 [19:54<00:00,  3.38s/it]

Saved batch 3520 to 3521
Done! Fully robust pipeline completed.
